# Dataset 6 — Individual Household Electric Power Consumption (UCI)

**Situação:** uma residência possui monitoramento elétrico detalhado e deseja identificar
episódios de **demanda elevada** que **também** apresentem **corrente acima do comportamento médio**.

O dataset é o mesmo usado como referência em aula, mas o critério de análise é diferente:
aqui o corte de potência é **75% do valor máximo** e existe uma **segunda condição** baseada
na corrente (`Global_intensity`).

**Integrantes do grupo:** _(preencher)_
**Data:** _(preencher)_

---

## Sobre a base original

| Item | Descrição |
|---|---|
| Arquivo | `household_power_consumption.txt` |
| Registros | 2.075.259 medições (dez/2006 a nov/2010, 1 medição por minuto) |
| Separador | ponto e vírgula (`;`) |
| Ausentes | marcados com `?` — cerca de 1,25% das linhas (~25.979 registros) |

| Atributo | Significado | Unidade |
|---|---|---|
| `Date`, `Time` | data e hora da medição | — |
| `Global_active_power` | potência ativa média por minuto | kW |
| `Global_reactive_power` | potência reativa média por minuto | kW |
| `Voltage` | tensão média por minuto | V |
| `Global_intensity` | **corrente** média por minuto | A |
| `Sub_metering_1` | cozinha (lava-louças, forno, micro-ondas) | Wh |
| `Sub_metering_2` | lavanderia (máquina de lavar, secadora, geladeira, luz) | Wh |
| `Sub_metering_3` | aquecedor de água + ar-condicionado | Wh |

---
# Etapa A — Orange Data Mining

Fluxo de widgets montado no canvas (nesta ordem):

```
File  →  Data Table  →  Impute  →  Select Columns  →  Data Sampler  →  Save Data
                ↑                                          ↓
          (inspeção dos                              Data Table
           valores ausentes)                     (conferência final)
```

### A.1 — File (carregar o dataset ORIGINAL)

Carregar `household_power_consumption.txt` (a base **completa**, não a amostra usada anteriormente).

Na janela do widget **File**:

- `Browse` → selecionar o arquivo `.txt`.
- Se o Orange não reconhecer o formato, renomear/salvar como `.csv` ou informar o separador `;`.
- Conferir na tabela inferior do widget o papel (*role*) e o tipo de cada coluna:
  - `Date` e `Time` → **meta / skip** (não entram na análise numérica);
  - as 7 demais colunas → **feature**, tipo **numeric**;
  - nenhum atributo deve ficar como *target*.
- O Orange interpreta `?` como valor ausente automaticamente. Se as colunas vierem como
  *text/categorical* por causa do `?`, forçar o tipo para **numeric** — as células `?` viram
  ausentes (`NaN`).

**Saída esperada:** ~2.075.259 instâncias e 7 atributos numéricos.

### A.2 — Data Table (inspeção dos ausentes)

Conectar `File → Data Table`. Na barra de status do widget aparece algo como
*"2075259 instances (25979 with missing values), 7 features"*.

Para visualizar melhor, também é possível usar o widget **Feature Statistics**, que mostra a
coluna *Missing* com o percentual de ausentes por atributo (**≈ 1,25%** em todas as sete
variáveis — os ausentes ocorrem em linhas inteiras).

> **Registrar no relatório:** número total de instâncias, número de instâncias com valores
> ausentes e o percentual correspondente.

### A.3 — Tratamento dos valores ausentes

Widget **Impute**, conectado após o `Data Table`.

Duas alternativas (usar a que foi trabalhada em aula):

| Opção | Configuração no Impute | Efeito |
|---|---|---|
| **(a) Remover** as linhas incompletas | *Default method* → **Remove instances with unknown values** | ficam ~2.049.280 registros |
| **(b) Substituir** pela média | *Default method* → **Average/Most frequent** | mantém 2.075.259 registros |

Como os ausentes representam apenas ~1,25% e ocorrem em **linhas inteiras** (não há informação
parcial a aproveitar), a **remoção (a)** é a opção mais indicada: imputar a média criaria
~26 mil registros artificiais e "achataria" a distribuição justamente na cauda que interessa
à análise (picos de potência).

Conferir o resultado ligando um novo **Data Table** na saída do `Impute`: a mensagem
*"with missing values"* deve desaparecer.

### A.4 — Select Columns

Manter apenas os 7 atributos pedidos, todos como **Features**:

`Global_active_power`, `Global_reactive_power`, `Voltage`, `Global_intensity`,
`Sub_metering_1`, `Sub_metering_2`, `Sub_metering_3`

`Date` e `Time` vão para **Ignored** (ou permanecem como *Metas*, se preferir manter a
rastreabilidade temporal). *Target* e *Metas* ficam vazios.

### A.5 — Data Sampler (amostra aleatória de 10%)

- *Sampling type*: **Fixed proportion of data** → **10%**
- Marcar **Replicable (deterministic) sampling** para que a amostra seja reproduzível
  pelo grupo inteiro.
- Clicar em **Sample Data**.
- Saída: ~204.928 instâncias (se os ausentes foram removidos).

⚠️ Atenção: o `Data Sampler` tem duas saídas — usar **Data Sample** (não *Remaining Data*).

### A.6 — Save Data (exportar CSV)

- Conectar `Data Sampler (Data Sample) → Save Data`.
- Formato: **Comma-separated values (.csv)**.
- Nome sugerido: **`amostra_10pct_tratada.csv`**.
- Salvar na mesma pasta deste notebook.

> Dica: se marcar *"Add type annotations"*, o Orange grava linhas extras de cabeçalho.
> Para facilitar a leitura no pandas, **desmarcar** essa opção.

---
# Etapa B — Python / Pandas

## B.1 — Carregar a amostra e simplificar os nomes dos atributos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Caminho do CSV exportado pelo Orange na Etapa A
CAMINHO = "amostra_10pct_tratada.csv"

# sep=None + engine="python" -> o pandas detecta sozinho se o separador e , ou ;
df = pd.read_csv(CAMINHO, sep=None, engine="python")

print("Dimensoes da amostra:", df.shape)
print("\nColunas originais:")
print(list(df.columns))
df.head()

In [ ]:
# ---- Renomeacao para nomes mais simples ----
mapa = {
    "global_active_power":   "pot_ativa",
    "global_reactive_power": "pot_reativa",
    "voltage":               "tensao",
    "global_intensity":      "corrente",
    "sub_metering_1":        "sub1",
    "sub_metering_2":        "sub2",
    "sub_metering_3":        "sub3",
}

# normaliza (minusculas, sem espacos) antes de aplicar o mapa
df.columns = [mapa.get(c.strip().lower().replace(" ", "_"), c.strip().lower())
              for c in df.columns]

# garante tipo numerico em todas as colunas de interesse
for c in df.columns:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("Colunas apos a renomeacao:")
print(list(df.columns))
df.head()

In [ ]:
# ---- Verificacao de qualidade: os ausentes ja foram tratados na Etapa A? ----
print("Valores ausentes por coluna:")
print(df.isna().sum())
print("\nTotal de registros:", len(df))

# Se ainda houver ausentes (por exemplo, se foi usada a imputacao no Orange
# ou se algo escapou na exportacao), a linha abaixo garante a limpeza:
df = df.dropna().reset_index(drop=True)
print("Registros apos dropna():", len(df))

In [ ]:
# Estatisticas descritivas gerais da amostra
df.describe().T

## B.2 — Valor máximo de potência ativa

In [ ]:
max_pot = df["pot_ativa"].max()

print(f"Potencia ativa MAXIMA da amostra: {max_pot:.4f} kW")
print(f"Potencia ativa MEDIA  da amostra: {df['pot_ativa'].mean():.4f} kW")
print(f"Potencia ativa MEDIANA da amostra: {df['pot_ativa'].median():.4f} kW")

## B.3 — Limite de 75% do máximo e DataFrame dos registros acima do limite

In [ ]:
limite_pot = 0.75 * max_pot
print(f"Limite = 75% do maximo = 0.75 x {max_pot:.4f} = {limite_pot:.4f} kW")

# DataFrame 1: apenas a condicao de potencia
df_alta_potencia = df[df["pot_ativa"] > limite_pot].copy()

print(f"\nRegistros acima do limite: {len(df_alta_potencia)}")
df_alta_potencia.head(10)

## B.4 — Quantidade e percentual de registros selecionados

In [ ]:
total = len(df)
qtd_1 = len(df_alta_potencia)
pct_1 = qtd_1 / total * 100

print(f"Total de registros na amostra .............. {total:,}")
print(f"Registros com pot_ativa > {limite_pot:.4f} kW ....... {qtd_1:,}")
print(f"Percentual do total ........................ {pct_1:.4f}%")

## B.5 — Corrente média da amostra

In [ ]:
corrente_media = df["corrente"].mean()

print(f"Corrente MEDIA da amostra ...... {corrente_media:.4f} A")
print(f"Corrente MEDIANA ............... {df['corrente'].median():.4f} A")
print(f"Corrente MAXIMA ................ {df['corrente'].max():.4f} A")
print(f"Corrente MINIMA ................ {df['corrente'].min():.4f} A")

# Quantos registros da amostra INTEIRA ficam acima da corrente media?
acima_corrente = (df["corrente"] > corrente_media).sum()
print(f"\nRegistros com corrente acima da media: {acima_corrente:,} "
      f"({acima_corrente/total*100:.2f}% do total)")

## B.6 — Segundo DataFrame: potência acima de 75% do máximo **E** corrente acima da média

In [ ]:
# DataFrame 2: as duas condicoes simultaneamente (operador & = E logico)
df_alta_dupla = df[(df["pot_ativa"] > limite_pot) &
                   (df["corrente"] > corrente_media)].copy()

qtd_2 = len(df_alta_dupla)
pct_2 = qtd_2 / total * 100

print(f"Registros com pot_ativa > {limite_pot:.4f} kW E corrente > "
      f"{corrente_media:.4f} A: {qtd_2:,}")
print(f"Percentual do total: {pct_2:.4f}%")
df_alta_dupla.head(10)

## B.7 — Comparação entre os dois conjuntos

In [ ]:
comparacao = pd.DataFrame({
    "Criterio": [
        "1 condicao: pot_ativa > 75% do maximo",
        "2 condicoes: pot_ativa > 75% do max E corrente > media",
    ],
    "Qtd. registros": [qtd_1, qtd_2],
    "% da amostra":   [pct_1, pct_2],
})
comparacao["% retido em relacao ao 1o filtro"] = [
    100.0,
    (qtd_2 / qtd_1 * 100) if qtd_1 else np.nan,
]
comparacao

In [ ]:
# Quantos registros o segundo filtro ELIMINOU?
eliminados = qtd_1 - qtd_2
print(f"Registros eliminados pela 2a condicao: {eliminados}")
if qtd_1:
    print(f"Reducao percentual: {eliminados/qtd_1*100:.4f}%")

# Existe algum registro de alta potencia com corrente ABAIXO da media?
menor_corrente_no_grupo = df_alta_potencia["corrente"].min()
print(f"\nMenor corrente dentro do grupo de alta potencia: "
      f"{menor_corrente_no_grupo:.4f} A")
print(f"Corrente media da amostra ........................ {corrente_media:.4f} A")
print(f"A menor corrente do grupo ja e maior que a media? "
      f"{menor_corrente_no_grupo > corrente_media}")

In [ ]:
# Por que isso acontece? -> potencia e corrente sao quase colineares.
# Relacao fisica: P (W) ~= V x I x cos(phi)  =>  I ~= P x 1000 / V

correl = df["pot_ativa"].corr(df["corrente"])
print(f"Correlacao de Pearson entre pot_ativa e corrente: {correl:.6f}")

df["corrente_estimada"] = df["pot_ativa"] * 1000 / df["tensao"]
erro = (df["corrente"] - df["corrente_estimada"]).abs()
print(f"Erro medio absoluto entre corrente medida e P/V estimada: {erro.mean():.4f} A")

# Corrente correspondente ao limite de potencia, com tensao tipica
tensao_media = df["tensao"].mean()
corrente_no_limite = limite_pot * 1000 / tensao_media
print(f"\nTensao media da amostra: {tensao_media:.2f} V")
print(f"Corrente equivalente ao limite de {limite_pot:.3f} kW: "
      f"~{corrente_no_limite:.2f} A")
print(f"Ou seja, cerca de {corrente_no_limite/corrente_media:.1f}x a corrente media "
      f"({corrente_media:.2f} A).")

In [ ]:
# Comparacao das estatisticas descritivas dos dois grupos
cols = ["pot_ativa", "pot_reativa", "tensao", "corrente", "sub1", "sub2", "sub3"]

resumo = pd.concat(
    [
        df[cols].mean().rename("Amostra completa"),
        df_alta_potencia[cols].mean().rename("Filtro 1 (potencia)"),
        df_alta_dupla[cols].mean().rename("Filtro 2 (potencia + corrente)"),
    ],
    axis=1,
)
resumo

In [ ]:
# Visualizacao: dispersao potencia x corrente com os dois cortes
fig, ax = plt.subplots(figsize=(9, 6))

amostra_plot = df.sample(min(20000, len(df)), random_state=42)
ax.scatter(amostra_plot["pot_ativa"], amostra_plot["corrente"],
           s=4, alpha=0.25, label="Amostra (10%)")
ax.scatter(df_alta_dupla["pot_ativa"], df_alta_dupla["corrente"],
           s=8, alpha=0.7, color="crimson", label="Potencia > 75% max E corrente > media")

ax.axvline(limite_pot, linestyle="--", color="black",
           label=f"75% do maximo = {limite_pot:.2f} kW")
ax.axhline(corrente_media, linestyle=":", color="darkgreen",
           label=f"Corrente media = {corrente_media:.2f} A")

ax.set_xlabel("Potencia ativa global (kW)")
ax.set_ylabel("Corrente global (A)")
ax.set_title("Relacao potencia ativa x corrente e os dois criterios de selecao")
ax.legend(loc="upper left", fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Distribuicao da potencia ativa e a posicao do corte
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df["pot_ativa"], bins=120, color="steelblue", alpha=0.85)
ax.axvline(limite_pot, linestyle="--", color="crimson", linewidth=2,
           label=f"Limite 75% do maximo = {limite_pot:.2f} kW")
ax.set_yscale("log")
ax.set_xlabel("Potencia ativa global (kW)")
ax.set_ylabel("Frequencia (escala log)")
ax.set_title("Distribuicao da potencia ativa - os episodios de pico sao raros")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Exportar os dois conjuntos para conferencia / anexo do relatorio
df_alta_potencia.to_csv("df_alta_potencia.csv", index=False)
df_alta_dupla.to_csv("df_alta_potencia_e_corrente.csv", index=False)
print("Arquivos gerados: df_alta_potencia.csv e df_alta_potencia_e_corrente.csv")

---
# Respostas interpretativas

> As tabelas acima trazem os valores exatos da **sua** amostra. Os números citados abaixo são
> os esperados para uma amostra aleatória de 10% desta base — confira e ajuste conforme a
> saída do seu notebook.

### 1. Qual o valor máximo de potência ativa e o limite adotado?

Na base completa o máximo é da ordem de **11,1 kW**; em uma amostra de 10% o máximo costuma
ficar entre **9 e 11 kW**. O limite de trabalho é 75% desse valor, algo em torno de
**7 a 8 kW** — um patamar que só é atingido quando vários equipamentos de alta demanda
(aquecedor de água, chuveiro/ar-condicionado, forno, máquina de lavar) operam ao mesmo tempo.

### 2. O que representa o percentual de registros selecionados?

O percentual costuma ficar **abaixo de 0,1%** da amostra. Isso confirma o formato da
distribuição vista no histograma: a potência ativa é fortemente **assimétrica à direita**
(média ~1,09 kW, mediana ~0,60 kW), com a maior parte do tempo em consumo baixo e uma cauda
longa e rara de picos. Portanto, os episódios de demanda elevada são **eventos excepcionais**,
não o comportamento típico da residência.

### 3. Qual o efeito de incluir a corrente como segunda condição?

**Praticamente nenhum: os dois conjuntos são iguais (ou quase).** A segunda condição não
elimina registros porque potência ativa e corrente **não são variáveis independentes** — elas
descrevem o mesmo fenômeno físico:

$$P \approx V \times I \times \cos\varphi \quad\Longrightarrow\quad I \approx \frac{P \times 1000}{V}$$

Como a tensão da rede é praticamente constante (≈ 240 V, com variação de poucos por cento),
a corrente é essencialmente uma **transformação linear** da potência ativa. A correlação de
Pearson calculada acima fica em torno de **0,999**.

O detalhe decisivo é a **posição relativa dos dois cortes**:

| Critério | Valor aproximado | Equivalente em corrente |
|---|---|---|
| Corrente média da amostra | ~4,6 A | ~4,6 A |
| Limite de 75% do máx. de potência | ~8 kW | **~33 A** |

Ou seja, o corte de potência exige uma corrente **cerca de 7 vezes maior** que a média.
Qualquer registro que passe pelo primeiro filtro **já passou, por construção física, pelo
segundo**. A condição de corrente é **redundante** neste recorte — ela está logicamente
contida na condição de potência.

### 4. Quando a segunda condição *seria* útil?

A inclusão da corrente só teria poder discriminante em cenários em que ela deixasse de ser
redundante, por exemplo:

- **Limites mais próximos**: se o corte de potência fosse a média (≈1,09 kW ↔ ≈4,5 A), os dois
  filtros ficariam quase sobrepostos e pequenas diferenças de fator de potência/tensão fariam
  os conjuntos divergirem.
- **Corrente com limite mais alto** (ex.: acima do 3º quartil ou de um percentil elevado),
  em vez da média.
- **Analisar a razão entre as variáveis** em vez de aplicar dois cortes independentes: por
  exemplo, procurar minutos com corrente alta e potência ativa comparativamente baixa —
  indício de **baixo fator de potência** (muita potência reativa), que é um problema real de
  qualidade de energia.
- **Substituir o E lógico pelo OU** (`|`), para capturar episódios anômalos em qualquer uma
  das duas dimensões.

### 5. Conclusão para o problema proposto

A residência apresenta um perfil de consumo com picos raros e bem definidos. Para identificar
"episódios de demanda elevada com corrente acima do comportamento médio", **bastaria o filtro
de potência ativa** — a condição de corrente confirma o diagnóstico, mas não acrescenta
capacidade de seleção. Do ponto de vista de mineração de dados, esse é um exemplo claro de
**atributos redundantes / multicolinearidade**: incluir as duas variáveis em um mesmo critério
(ou em um mesmo modelo) adiciona custo sem adicionar informação.